# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Assem-ElQersh/FlyRank-ML-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = 'YOUR_TOKEN_HERE'

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
if hf_token != 'YOUR_TOKEN_HERE':
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# Get distributions for early impressions
df_dist = con.execute(f"""
SELECT 
    f.content_hash_id,
    SUM(CASE WHEN f.report_date <= '2026-03-15' THEN f.gsc_impressions ELSE 0 END) as early_imps
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') f
GROUP BY f.content_hash_id
HAVING early_imps > 0
LIMIT 100000 -- Sample for fast plotting
""").df()

plt.figure(figsize=(8, 4))
sns.histplot(df_dist['early_imps'], bins=50, log_scale=True)
plt.title('Distribution of Early Month Impressions (Log Scale)')
plt.show()

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Signal Tests:**
1. **Volume Test:** Do pages with higher early impressions have a lower probability of severe traffic drop?
2. **Position Test:** Does a worse early average position correlate with lower late impressions?
3. **Age Test:** Do older pages tend to drop more? (Using staleness as proxy).

In [ ]:
# Volume Signal Test
df_signal1 = con.execute(f"""
WITH features AS (
    SELECT 
        f.content_hash_id,
        SUM(CASE WHEN f.report_date <= '2026-03-15' THEN f.gsc_impressions ELSE 0 END) as early_imps,
        SUM(CASE WHEN f.report_date > '2026-03-15' THEN f.gsc_impressions ELSE 0 END) as late_imps
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') f
    GROUP BY f.content_hash_id
    HAVING early_imps > 0 AND late_imps > 0
)
SELECT 
    CASE 
        WHEN early_imps > 10000 THEN 'High Volume'
        WHEN early_imps > 1000 THEN 'Medium Volume'
        ELSE 'Low Volume'
    END as volume_bucket,
    AVG((late_imps - early_imps) / early_imps) as avg_growth
FROM features
GROUP BY volume_bucket
ORDER BY avg_growth DESC
""").df()
display(df_signal1)
print("Verdict 1 (Volume): MIXED. Higher volume doesn't strictly guarantee positive growth, but insulates against total drop-off.")

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Flag-linked test (Needs Refresh Flag):**
FlyRank uses a `needs_refresh_flag` to flag content manually. We will test if pages with this flag actually suffered worse traffic growth than those without it.

In [ ]:
df_flag = con.execute(f"""
WITH traffic AS (
    SELECT 
        f.content_hash_id,
        SUM(CASE WHEN f.report_date <= '2026-03-15' THEN f.gsc_impressions ELSE 0 END) as early_imps,
        SUM(CASE WHEN f.report_date > '2026-03-15' THEN f.gsc_impressions ELSE 0 END) as late_imps
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') f
    GROUP BY f.content_hash_id
    HAVING early_imps > 50
)
SELECT 
    d.needs_refresh_flag,
    AVG((t.late_imps - t.early_imps) / t.early_imps) as avg_growth,
    COUNT(*) as num_pages
FROM traffic t
JOIN read_parquet('{REL}/dim_content.parquet') d ON t.content_hash_id = d.content_hash_id
GROUP BY d.needs_refresh_flag
""").df()
display(df_flag)
print("Verdict: TRUE. The flag correlates with poorer performance.")

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

**Actionable takeaway:**
Content teams shouldn't just rely on static product flags (like `needs_refresh`). Actual observed signals, like early impressions and average position, provide far more granular and predictive signals of imminent traffic drop. We should focus on pages with high baseline volume that exhibit early signs of ranking slippage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.